# Huấn luyện Qwen 7B bằng QLoRA trên Google Colab

Notebook này được tạo ra để tự động hóa toàn bộ quá trình setup môi trường và chạy file huấn luyện mô hình ngôn ngữ lớn (LLM) của dự án `AI_race`.

**Yêu cầu trước khi chạy:**
1. Đảm bảo toàn bộ thư mục `AI_race` đã được upload lên Google Drive của bạn (mặc định để ở thư mục gốc `My Drive`).
2. Đảm bảo Notebook này đang được bật GPU: **Runtime (Thời gian chạy) > Change runtime type (Thay đổi loại thời gian chạy) > Hardware accelerator (Trình tăng tốc phần cứng) > Chọn L4 GPU**.

## 1. Kết nối với Google Drive
Chạy ô này để cấp quyền cho Colab truy cập vào Drive của bạn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Truy cập vào thư mục dự án
Nếu bạn để thư mục dự án ở một đường dẫn khác, hãy sửa lại đường dẫn bên dưới nhé.

In [ ]:
%cd "/content/drive/MyDrive/AI_race"

## 3. Cài đặt các thư viện cần thiết
Cài đặt các thư viện chuyên dụng để huấn luyện mô hình (HuggingFace Transformers, PEFT, TRL, BitsAndBytes) và lượng tử hóa (AutoAWQ).

In [ ]:
!pip install -q transformers==4.44.0 peft==0.12.0 trl==0.10.1 datasets==2.21.0 accelerate==0.34.2 autoawq
!pip install -q -U bitsandbytes

## 4. Chạy Script Huấn luyện (Fine-tune QLoRA)
Trọng số của mô hình sau khi học xong sẽ tự động được lưu đè lên Drive của bạn tại: `models/qwen_7b_medical_adapter`.

In [ ]:
!python phase_2_LLM/train/train_qlora.py

## 5. Đánh giá mô hình TRƯỚC khi Lượng tử hóa
Chạy file evaluate để xem chất lượng sinh text của mô hình (Sử dụng Base model + LoRA Adapter).

In [ ]:
!python phase_2_LLM/test/evaluate_model.py

## 6. Ép xung / Lượng tử hóa (Quantization AWQ 4-bit)
Nén toàn bộ model từ dạng Adapter phân mảnh sang dạng nguyên khối siêu nhẹ 4-bit (chỉ nặng khoảng 4.5 GB) để nhét vừa các server thi đấu.

In [ ]:
!python phase_2_LLM/quantization/quantize_awq.py

## 7. Đánh giá mô hình SAU khi Lượng tử hóa
Chạy file kiểm tra lại lần cuối để đảm bảo mô hình sau khi bị nén (ép xung) vẫn giữ được chất lượng trích xuất giống hệt bản gốc.

In [ ]:
!python phase_2_LLM/test/evaluate_awq_model.py